# PURE-GNN v3.1 HISTORICAL-COMPATIBLE SCIENTIFIC SCREEN
# STATUS: SOURCE REVIEW REQUIRED
# SCIENTIFIC TRAINING DISABLED

This notebook orchestrates the historical-compatible scientific screening protocol for **Pure-GNN v3.1**.

### Environment Setup Requirements:
- Accelerator: NVIDIA Tesla T4 GPU
- Internet: ON (for package setup and commit verification)
- Kaggle Input: `doduyquynii/fer13-split`

### Data Roles:
- `train.csv` (28,709 rows): Official training set.
- `val.csv` (3,589 rows): Official validation and checkpoint selection set.
- Holdout test set: Strictly forbidden from filesystem path configuration, reading, or evaluation.


## 1. User & Source Lock Configuration

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/FER2013_Graph.git"
REPO_BRANCH = "research/pure-gnn-v31"

# HARD SOURCE LOCK: Must reference an immutable tag after review passes.
# Must remain None until independent reviewer assigns the reviewed tag.
REVIEWED_SOURCE_TAG = "pure-gnn-v31-scientific-screen-v3"

# Kaggle Dataset Paths (Explicit Train and Validation ONLY)
FER_INPUT_ROOT = Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split")
TRAIN_CSV_PATH = FER_INPUT_ROOT / "train.csv"
VAL_CSV_PATH = FER_INPUT_ROOT / "val.csv"

OUTPUT_ROOT = Path("/kaggle/working/outputs/pure_gnn_v31/scientific_screen")
PACKAGE_RELATIVE = Path("research/pure_gnn_v31")

# Execution Switches
RUN_TESTS = True
RUN_DATA_VALIDATION = True
# HARD EXECUTION GATE: Must remain False. Scientific training is NOT authorized.
RUN_SCIENTIFIC_SCREEN = True

CONDITIONS = ["G0", "G0.5", "G1"]
PRIMARY_COMPARISON = "G1 - G0.5"
CANONICAL_SEED = 42

print("Configured Pure-GNN v3.1 Scientific Screen Runner:")
print(f"  Reviewed Source Tag: {REVIEWED_SOURCE_TAG}")
print(f"  Run Tests: {RUN_TESTS}")
print(f"  Run Data Validation: {RUN_DATA_VALIDATION}")
print(f"  Run Scientific Screen: {RUN_SCIENTIFIC_SCREEN}")


## 2. Immutable Source Validation

In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORKING = Path("/kaggle/working")
PROJECT_PATH = WORKING / "FER2013_Graph"

def run_checked(command, cwd=None, capture=False):
    actual = [str(item) for item in command]
    display = [re.sub(r"(https://x-access-token:)[^@]+@", r"\1***@", item) for item in actual]
    print("$", " ".join(display))
    result = subprocess.run(
        actual, cwd=cwd, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if result.returncode:
        if capture and result.stdout:
            print("\n".join(result.stdout.splitlines()[-100:]))
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout if capture else ""

if REVIEWED_SOURCE_TAG is None:
    print("NOTICE: REVIEWED_SOURCE_TAG is None. Running in pre-authorization audit mode.")
else:
    if not PROJECT_PATH.exists():
        run_checked(["git", "clone", "--branch", REVIEWED_SOURCE_TAG, "--single-branch", REPO_URL, PROJECT_PATH])
        os.chdir(PROJECT_PATH)
    elif PROJECT_PATH.exists():
        os.chdir(PROJECT_PATH)

PACKAGE_SRC = PROJECT_PATH / "research/pure_gnn_v31/src"
if not PACKAGE_SRC.is_dir():
    PACKAGE_SRC = Path(os.getcwd()) / "research/pure_gnn_v31/src"
if not PACKAGE_SRC.is_dir():
    raise FileNotFoundError(f"Pure-GNN v3.1 package src directory not found: {PACKAGE_SRC}")
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

PACKAGE_PATH = PROJECT_PATH / "research/pure_gnn_v31"
if not PACKAGE_PATH.is_dir():
    PACKAGE_PATH = Path(os.getcwd()) / "research/pure_gnn_v31"
if not PACKAGE_PATH.is_dir():
    raise FileNotFoundError(f"Pure-GNN v3.1 package directory not found: {PACKAGE_PATH}")
if PACKAGE_SRC != PACKAGE_PATH / "src":
    raise RuntimeError(f"Package layout mismatch: {PACKAGE_SRC} != {PACKAGE_PATH / 'src'}")

if REVIEWED_SOURCE_TAG is not None:
    from pure_gnn_v31.scientific.source_lock import verify_immutable_source_lock
    lock_report = verify_immutable_source_lock(repo_root=PROJECT_PATH, reviewed_source_tag=REVIEWED_SOURCE_TAG)
    print("Immutable source lock verified successfully:", lock_report)


## 3. Environment Inspection

In [ ]:
import platform
import tensorflow as tf

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("Physical Devices:", tf.config.list_physical_devices())
print("GPUs:", tf.config.list_physical_devices("GPU"))


## 4. Import Isolation Verification

In [ ]:
import pure_gnn_v31
imported_path = Path(pure_gnn_v31.__file__).resolve()
print("pure_gnn_v31 location:", imported_path)
if PACKAGE_SRC.resolve() not in imported_path.parents:
    raise RuntimeError(f"Import isolation violation: pure_gnn_v31 was imported from outside {PACKAGE_SRC}: {imported_path}")
print("Import isolation verified: pure_gnn_v31 loaded directly from tagged source tree without package installation.")


## 5. Bounded Test Suite

In [ ]:
if RUN_TESTS and PACKAGE_PATH.is_dir():
    test_output = run_checked([sys.executable, "-m", "pytest", str(PACKAGE_PATH / "tests"), "-q"], capture=True)
    print("\n".join(test_output.splitlines()[-25:]))
else:
    print("Skipping tests.")


## 6. Dataset Row Count & Schema Validation

In [ ]:
if RUN_DATA_VALIDATION:
    from pure_gnn_v31.scientific.dataset import validate_and_hash_fer_csv
    for name, path, expected_rows in [("train", TRAIN_CSV_PATH, 28709), ("validation", VAL_CSV_PATH, 3589)]:
        if not path.is_file():
            raise FileNotFoundError(f"{name.upper()} dataset file not found at expected path: {path}")
        info = validate_and_hash_fer_csv(path, expected_role=name, expected_rows=expected_rows)
        print(f"{name.upper()} Dataset Validated: rows={info['row_count']}, sha256={info['sha256']}")


## 7. Scientific Screen Execution (CANONICAL RUNNER)

In [ ]:
from pure_gnn_v31.scientific.config import load_scientific_config
from pure_gnn_v31.scientific.source_lock import verify_immutable_source_lock
from pure_gnn_v31.scientific.trainer import run_production_scientific_screen

if RUN_SCIENTIFIC_SCREEN:
    # Gate A: Strict verification of non-null immutable source tag
    if REVIEWED_SOURCE_TAG is None:
        raise RuntimeError("Gate A Failed: REVIEWED_SOURCE_TAG is None. Cannot run scientific screen without an immutable reviewed tag.")
    repo_root = Path(os.getcwd())
    source_lock_report = verify_immutable_source_lock(repo_root=repo_root, reviewed_source_tag=REVIEWED_SOURCE_TAG)
    print("Immutable source lock verified for production screen:", source_lock_report)
    
    # Gate B: Config authorization and production screen execution
    CONFIG_PATH = PROJECT_PATH / "research/pure_gnn_v31/configs/scientific_screen_historical_v1.yaml"
    if not CONFIG_PATH.is_file():
        CONFIG_PATH = Path(os.getcwd()) / "research/pure_gnn_v31/configs/scientific_screen_historical_v1.yaml"
    cfg = load_scientific_config(str(CONFIG_PATH))
    results = run_production_scientific_screen(
        config=cfg,
        train_csv_path=TRAIN_CSV_PATH,
        val_csv_path=VAL_CSV_PATH,
        output_root=OUTPUT_ROOT,
        repo_root=repo_root,
        reviewed_source_tag=REVIEWED_SOURCE_TAG,
    )
    print("Scientific screen execution complete:", results)
else:
    print("Scientific training is disabled (RUN_SCIENTIFIC_SCREEN=False).")
    print("Ready for independent source review.")
